In [ ]:
# ============================================================
# MAPCT-v10: GRAPH NEURAL NETWORK FOR REGIMEN GENERATION
# With Semantic & Structured Mechanism Integration
#
# Architecture based on state-of-the-art biomedical GNNs:
# - DruGNNosis-MoA: GNN with LLM embeddings [citation:1]
# - PT-KGNN: Pre-training biomedical knowledge graphs [citation:2]
# - PKGCN: Pharmacological knowledge graph networks [citation:3]
# - FuseLinker: LLM + GNN for biomedical link prediction [citation:10]
# ============================================================
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.2.0+cu121.html

import os
import re
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from typing import Dict, List, Tuple, Optional
import hashlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_mean_pool
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader as PyGDataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



Looking in links: https://data.pyg.org/whl/torch-2.2.0+cu121.html
Using device: cpu


In [ ]:
# ============================================================
# SECTION 0: DRUG NAME SORTING & NORMALIZATION
# ============================================================

def normalize_drug_name(name: str) -> str:
    """Normalize drug name by removing brand info and standardizing format"""
    if not isinstance(name, str):
        return ""
    name = name.lower().strip()
    name = re.sub(r"\([^)]*\)", "", name)
    name = re.sub(r"\s+brand:.*", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s*\+\s*", ", ", name)
    name = re.sub(r"\s*,\s*", ", ", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()


def sort_drug_combo(combo: str) -> str:
    """Sort drugs within a combination alphabetically"""
    if not isinstance(combo, str):
        return ""
    drugs = [d.strip() for d in combo.split(",") if d.strip()]
    drugs = sorted(drugs)
    return ", ".join(drugs)


def split_combo(combo: str) -> List[str]:
    """Split a combo string into individual drug names"""
    if not isinstance(combo, str):
        return []
    return [d.strip() for d in combo.split(",") if d.strip()]


def deduplicate_combo(combo: str) -> str:
    """Remove duplicate drugs within a combination"""
    drugs = split_combo(combo)
    unique_drugs = sorted(set(drugs))
    return ", ".join(unique_drugs)





In [ ]:
# ============================================================
# SECTION 1: DATA LOADING
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

file_path1 = '/content/drive/MyDrive/khezri/drug_data/merged_drug.xlsx'
file_path2 = '/content/drive/MyDrive/khezri/drug_data/drug_mechanism.xlsx'

clinic_df = pd.read_excel(file_path1, engine="openpyxl")
mech_df = pd.read_excel(file_path2, engine="openpyxl")

print(f"Loaded {len(clinic_df)} patients, {len(mech_df)} mechanism entries")

# Apply normalization and sorting
clinic_df["merged_drugs_norm"] = (
    clinic_df["merged_drugs"]
    .astype(str)
    .apply(normalize_drug_name)
    .apply(sort_drug_combo)
    .apply(deduplicate_combo)
)

mech_df["Drug Name Norm"] = (
    mech_df["Drug Name"]
    .astype(str)
    .apply(normalize_drug_name)
    .apply(sort_drug_combo)
    .apply(deduplicate_combo)
)

print(f"✅ Drug names normalized")
print(f"   Unique regimens: {clinic_df['merged_drugs_norm'].nunique()}")

ValueError: mount failed

In [ ]:
# ============================================================
# SECTION 2: LEAKAGE‑FREE SPLIT BY REGIMEN (GROUP‑BASED)
# ============================================================
from sklearn.model_selection import GroupShuffleSplit

# Groups = unique drug combinations (normalized)
groups = clinic_df["merged_drugs_norm"].values

# Step 1: Split off test set (20% of groups)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx_train_val, idx_test = next(gss1.split(clinic_df, groups=groups))

# Step 2: From remaining groups, split off validation (20% of those groups → 16% of total patients)
groups_train_val = groups[idx_train_val]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx_train_rel, idx_val_rel = next(gss2.split(clinic_df.iloc[idx_train_val], groups=groups_train_val))

# Convert relative indices to original indices
idx_train = idx_train_val[idx_train_rel]
idx_val = idx_train_val[idx_val_rel]

print(f"Split sizes: Train={len(idx_train)}, Val={len(idx_val)}, Test={len(idx_test)}")

train_df = clinic_df.iloc[idx_train].copy()
val_df = clinic_df.iloc[idx_val].copy()
test_df = clinic_df.iloc[idx_test].copy()

print(f"Splits: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")


# ============================================================
# SECTION 3: DRUG VOCABULARY (TRAIN ONLY)
# ============================================================

def build_drug_vocabulary(train_df):
    all_drugs = set()
    for combo in train_df["merged_drugs_norm"]:
        for drug in split_combo(combo):
            all_drugs.add(drug)
    drug_to_idx = {drug: i for i, drug in enumerate(sorted(all_drugs))}
    idx_to_drug = {i: drug for drug, i in drug_to_idx.items()}
    return drug_to_idx, idx_to_drug, len(drug_to_idx)

def encode_regimen_multi_hot(combo, drug_to_idx):
    vec = np.zeros(len(drug_to_idx), dtype=np.float32)
    for drug in split_combo(combo):
        if drug in drug_to_idx:
            vec[drug_to_idx[drug]] = 1.0
    return vec

def filter_to_train_vocab(combo, drug_to_idx):
    if not isinstance(combo, str):
        return ""
    drugs = [d.strip() for d in combo.split(",") if d.strip()]
    filtered = [d for d in drugs if d in drug_to_idx]
    return ", ".join(sorted(filtered))

drug_to_idx, idx_to_drug, num_drugs = build_drug_vocabulary(train_df)
drug_vocab = list(drug_to_idx.keys())
print(f"Drug vocabulary size: {num_drugs}")

val_df["merged_drugs_norm"] = val_df["merged_drugs_norm"].apply(lambda x: filter_to_train_vocab(x, drug_to_idx))
test_df["merged_drugs_norm"] = test_df["merged_drugs_norm"].apply(lambda x: filter_to_train_vocab(x, drug_to_idx))


# ============================================================
# SECTION 4: CLINICAL PREPROCESSING
# ============================================================

scaler = StandardScaler()
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_num_train = scaler.fit_transform(train_df[numeric_cols])
X_cat_train = ohe.fit_transform(train_df[categorical_cols])
X_train = np.hstack([X_num_train, X_cat_train])

X_num_val = scaler.transform(val_df[numeric_cols])
X_cat_val = ohe.transform(val_df[categorical_cols])
X_val = np.hstack([X_num_val, X_cat_val])

X_num_test = scaler.transform(test_df[numeric_cols])
X_cat_test = ohe.transform(test_df[categorical_cols])
X_test = np.hstack([X_num_test, X_cat_test])

print(f"Clinical feature dimension: {X_train.shape[1]}")


# ============================================================
# SECTION 5: SEMANTIC + STRUCTURED MECHANISM EMBEDDINGS
# ============================================================

from sentence_transformers import SentenceTransformer

def stable_hash(x: str, mod: int) -> int:
    return int(hashlib.md5(x.encode()).hexdigest(), 16) % mod


class StructuredMechanismEncoder:
    """Convert structured mechanism fields to embeddings"""
    def __init__(self, struct_dim=128):
        self.struct_dim = struct_dim
        self.n_fields = 8
        self.slice_len = struct_dim // self.n_fields

        self.offsets = {
            "chembl": 0,
            "target_ids": self.slice_len,
            "uniprot_ids": 2 * self.slice_len,
            "mesh_ids": 3 * self.slice_len,
            "pathway_ids": 4 * self.slice_len,
            "atc_codes": 5 * self.slice_len,
            "pharm_class": 6 * self.slice_len,
            "mechanism_ids": 7 * self.slice_len,
        }

        self.field_columns = {
            "chembl": "ChEMBL ID",
            "target_ids": "Target IDs",
            "uniprot_ids": "UniProt Target ID(s)",
            "mesh_ids": "MeSH Mechanism ID(s)",
            "pathway_ids": "Pathway IDs",
            "atc_codes": "ATC Code(s)",
            "pharm_class": "Pharmacologic Class",
            "mechanism_ids": "Mechanism IDs",
        }

    def encode_field(self, values, field_name):
        vector = torch.zeros(self.slice_len)
        if values is None:
            return vector
        if isinstance(values, pd.Series):
            values = values.dropna().tolist()
        elif isinstance(values, str):
            values = [v.strip() for v in values.split(";") if v.strip()]
        elif not isinstance(values, (list, tuple)):
            values = [values]

        for v in values:
            v_str = str(v).strip()
            if v_str and v_str not in ["", "nan", "None"]:
                idx = stable_hash(v_str, self.slice_len)
                vector[idx] += 1.0
        return vector

    def encode(self, row):
        vector = torch.zeros(self.struct_dim)
        for field_name, col_name in self.field_columns.items():
            if col_name in row.index:
                vals = row[col_name]
                field_vec = self.encode_field(vals, field_name)
                offset = self.offsets[field_name]
                vector[offset:offset + self.slice_len] = field_vec
        if torch.norm(vector) > 0:
            vector = vector / torch.norm(vector)
        return vector


def build_mechanism_embeddings(train_drugs, mech_df, embed_dim=128):
    """Build semantic + structured mechanism embeddings"""
    print("\n" + "="*70)
    print("🔬 BUILDING MECHANISM EMBEDDINGS (Semantic + Structured)")
    print("="*70)

    # Load semantic model
    semantic_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
    sem_dim = semantic_model.get_sentence_embedding_dimension()
    print(f"   Semantic dimension: {sem_dim}")

    # Structured encoder
    struct_encoder = StructuredMechanismEncoder(struct_dim=embed_dim)

    embeddings = {}
    missing_drugs = []

    for drug in tqdm(train_drugs, desc="Building embeddings"):
        row = mech_df[mech_df["Drug Name Norm"] == drug]

        if len(row) > 0:
            row = row.iloc[0]

            # Structured encoding
            struct_vec = struct_encoder.encode(row)

            # Semantic encoding
            text_cols = ["Mechanisms", "Description", "Target IDs", "Pathway IDs", "ATC Code(s)"]
            text_parts = []
            for col in text_cols:
                if col in row and pd.notna(row[col]):
                    text_parts.append(str(row[col]))

            if text_parts:
                full_text = " | ".join(text_parts)
                with torch.no_grad():
                    sem_vec = semantic_model.encode(full_text, convert_to_tensor=True)
                    sem_vec = F.normalize(sem_vec, dim=0).cpu()
            else:
                sem_vec = torch.zeros(sem_dim)

            # Fuse structured and semantic
            fused = torch.cat([struct_vec, sem_vec], dim=0)
            if torch.norm(fused) > 0:
                fused = fused / torch.norm(fused)

            embeddings[drug] = fused
        else:
            embeddings[drug] = torch.zeros(embed_dim + sem_dim)
            missing_drugs.append(drug)

    if missing_drugs:
        print(f"   ⚠️ No mechanism data for {len(missing_drugs)} drugs")

    print(f"✅ Built embeddings for {len(embeddings)} drugs")
    print(f"   Embedding dimension: {embed_dim + sem_dim}")

    return embeddings, embed_dim + sem_dim

train_drugs = set(drug_to_idx.keys())
drug_embeddings, full_embed_dim = build_mechanism_embeddings(train_drugs, mech_df, embed_dim=128)
print(f"Full embedding dimension: {full_embed_dim}")


# ============================================================
# SECTION 6: BUILD DRUG-DRUG INTERACTION GRAPH
# ============================================================
def build_drug_similarity_graph(drug_embeddings, drug_vocab, similarity_threshold=0.5, min_edges_per_node=1):
    """
    Build a drug similarity graph where edges connect drugs with similar mechanisms.
    This creates a GNN-compatible graph structure [citation:2][citation:5].
    """
    print("\n" + "="*70)
    print("📊 BUILDING DRUG SIMILARITY GRAPH")
    print("="*70)

    n_drugs = len(drug_vocab)
    drug_to_idx = {drug: i for i, drug in enumerate(drug_vocab)}

    # Precompute embeddings tensor
    emb_list = []
    for drug in drug_vocab:
        emb = drug_embeddings.get(drug, torch.zeros(full_embed_dim))
        emb_list.append(emb)
    embeddings_tensor = torch.stack(emb_list)  # [n_drugs, embed_dim]

    # Create edge list based on mechanism similarity
    edge_index = []
    edge_weights = []

    # Use efficient matrix computation for large vocab
    if n_drugs <= 500:  # Threshold for dense computation
        # Compute similarity matrix
        embeddings_norm = F.normalize(embeddings_tensor, dim=1)
        sim_matrix = torch.mm(embeddings_norm, embeddings_norm.t())

        # Get upper triangle indices
        for i in range(n_drugs):
            for j in range(i + 1, n_drugs):
                sim = sim_matrix[i, j].item()
                if sim > similarity_threshold:
                    edge_index.append([i, j])
                    edge_index.append([j, i])
                    edge_weights.append(sim)
                    edge_weights.append(sim)
    else:
        # Sparse computation for large vocab
        for i, drug1 in enumerate(drug_vocab):
            emb1 = embeddings_tensor[i]
            for j, drug2 in enumerate(drug_vocab[i+1:], i+1):
                emb2 = embeddings_tensor[j]
                sim = torch.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0)).item()

                if sim > similarity_threshold:
                    edge_index.append([i, j])
                    edge_index.append([j, i])
                    edge_weights.append(sim)
                    edge_weights.append(sim)

    # Ensure each node has at least one edge (add self-loops if needed)
    if len(edge_index) == 0:
        print("   ⚠️ No edges found. Adding self-loops for all nodes.")
        for i in range(n_drugs):
            edge_index.append([i, i])
            edge_weights.append(1.0)

    # Convert to tensors
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_weights = torch.tensor(edge_weights, dtype=torch.float32)

    print(f"   Created graph with {n_drugs} nodes and {edge_index.shape[1]} edges")
    print(f"   Average degree: {edge_index.shape[1] / n_drugs:.2f}")

    return edge_index, edge_weights, drug_to_idx


# Build drug similarity graph
drug_edge_index, drug_edge_weights, drug_to_graph_idx = build_drug_similarity_graph(
    drug_embeddings, drug_vocab, similarity_threshold=0.5
)


# ============================================================
# SECTION 7: GNN MODELS FOR MAPCT-v10
# ============================================================

class DrugMechanismGNN(nn.Module):
    """
    Graph Neural Network for encoding drug mechanisms
    Supports multiple GNN layer types: GCN, GAT, GraphSAGE [citation:3][citation:10]
    """
    def __init__(self, node_dim=full_embed_dim, hidden_dim=128, latent_dim=128,
                 n_layers=3, gnn_type='gat', n_heads=4, dropout=0.2):
        super().__init__()

        self.node_dim = node_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.gnn_type = gnn_type

        # Input projection
        self.input_proj = nn.Linear(node_dim, hidden_dim)

        # GNN layers
        self.convs = nn.ModuleList()
        for i in range(n_layers):
            if gnn_type == 'gcn':
                self.convs.append(GCNConv(hidden_dim, hidden_dim))
            elif gnn_type == 'gat':
                self.convs.append(GATConv(hidden_dim, hidden_dim, heads=n_heads, concat=False))
            elif gnn_type == 'sage':
                self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            else:
                raise ValueError(f"Unknown GNN type: {gnn_type}")

            self.bns = nn.ModuleList([nn.BatchNorm1d(hidden_dim) for _ in range(n_layers)])

        self.dropout = nn.Dropout(dropout)

        # Output projection
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, latent_dim),
            nn.BatchNorm1d(latent_dim),
            nn.ReLU()
        )

    def forward(self, x, edge_index, edge_weight=None):
        """
        Forward pass through GNN
        x: Node features [num_nodes, node_dim]
        edge_index: Graph edges [2, num_edges]
        """
        # Ensure edge_index is on same device and valid
        edge_index = edge_index.to(x.device)

        # Validate edge indices
        num_nodes = x.shape[0]
        max_idx = edge_index.max().item()
        if max_idx >= num_nodes:
            # Filter invalid edges
            valid_mask = (edge_index[0] < num_nodes) & (edge_index[1] < num_nodes)
            edge_index = edge_index[:, valid_mask]
            if edge_weight is not None:
                edge_weight = edge_weight[valid_mask]

        x = self.input_proj(x)

        for i, conv in enumerate(self.convs):
            if edge_weight is not None and edge_weight.shape[0] == edge_index.shape[1]:
                x_new = conv(x, edge_index, edge_weight)
            else:
                x_new = conv(x, edge_index)

            x_new = self.bns[i](x_new)
            x_new = F.relu(x_new)
            x_new = self.dropout(x_new)
            x = x + x_new  # Residual connection

        return F.normalize(self.output_proj(x), dim=1)



class ClinicalEncoder(nn.Module):
    """Encodes clinical features into latent space"""
    def __init__(self, input_dim, latent_dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, latent_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=1)


class RegimenMechanismEncoder(nn.Module):
    """
    Encodes regimen mechanism tensor into latent space using GNN-aggregated
    drug embeddings [citation:5]
    """
    def __init__(self, drug_gnn, latent_dim=128, max_drugs=10):
        super().__init__()
        self.drug_gnn = drug_gnn
        self.max_drugs = max_drugs
        self.pool_proj = nn.Linear(latent_dim, latent_dim)

    def forward(self, mech_tensor, mask, drug_graph_edge_index, drug_graph_weights=None):
        """
        mech_tensor: [batch, max_drugs, embed_dim] - drug mechanism embeddings
        mask: [batch, max_drugs] - which drugs are present
        """
        batch_size = mech_tensor.shape[0]

        # First, get GNN-enhanced drug embeddings
        all_drug_embeddings = self.drug_gnn(mech_tensor.view(-1, mech_tensor.shape[-1]),
                                             drug_graph_edge_index, drug_graph_weights)
        all_drug_embeddings = all_drug_embeddings.view(batch_size, self.max_drugs, -1)

        # Masked pooling over drugs in regimen
        mask_expanded = mask.unsqueeze(-1).float()
        pooled = (all_drug_embeddings * mask_expanded).sum(dim=1)
        pooled = pooled / mask_expanded.sum(dim=1).clamp(min=1)

        return F.normalize(self.pool_proj(pooled), dim=1)


class DrugDecoder(nn.Module):
    """Decodes latent to drug multi-hot predictions"""
    def __init__(self, latent_dim=128, num_drugs=None, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_drugs)
        )

    def forward(self, z):
        return self.net(z)


class MAPCTv10GNN(nn.Module):
    """
    MAPCT-v10: Graph Neural Network for Regimen Generation
    Integrates:
    1. Clinical encoder (patient data)
    2. Drug mechanism GNN (drug-drug similarity graph) [citation:5]
    3. Regimen mechanism encoder (Transformer/GNN for drug combinations)
    4. Contrastive learning between clinical and regimen spaces
    """
    def __init__(self, clin_dim, num_drugs, drug_graph_edge_index, drug_graph_weights,
                 latent_dim=128, drug_embed_dim=full_embed_dim,
                 gnn_type='gat', n_gnn_layers=3):
        super().__init__()

        # Register drug graph as buffer
        self.register_buffer('drug_graph_edge_index', drug_graph_edge_index)
        self.register_buffer('drug_graph_weights', drug_graph_weights)

        # Drug mechanism GNN (shared across all drugs)
        self.drug_gnn = DrugMechanismGNN(
            node_dim=drug_embed_dim,
            hidden_dim=latent_dim,
            latent_dim=latent_dim,
            n_layers=n_gnn_layers,
            gnn_type=gnn_type
        )

        # Clinical encoder
        self.clinical_encoder = ClinicalEncoder(clin_dim, latent_dim)

        # Regimen mechanism encoder (uses GNN-enhanced drug embeddings)
        self.regimen_encoder = RegimenMechanismEncoder(self.drug_gnn, latent_dim)

        # Drug decoder
        self.drug_decoder = DrugDecoder(latent_dim, num_drugs=num_drugs)



    def forward(self, clinical, mech_tensor, mech_mask):
        """
        Forward pass
        """
        # Encode clinical features
        z_clin = self.clinical_encoder(clinical)

        # Encode regimen mechanisms (using GNN)
        z_reg = self.regimen_encoder(mech_tensor, mech_mask,
                                      self.drug_graph_edge_index,
                                      self.drug_graph_weights)

        # Decode to drug predictions
        logits = self.drug_decoder(z_clin)

        return logits, z_clin, z_reg

    def get_drug_embeddings(self, drug_indices=None):
        """Get GNN-enhanced embeddings for specific drugs"""
        if drug_indices is None:
            # Return all drug embeddings
            dummy_mech = torch.zeros(len(drug_vocab), 1, full_embed_dim).to(device)
            return self.drug_gnn(dummy_mech.squeeze(1),
                                  self.drug_graph_edge_index,
                                  self.drug_graph_weights)
        else:
            # Return embeddings for specific drugs
            drug_mech = torch.stack([drug_embeddings[drug_vocab[i]] for i in drug_indices])
            return self.drug_gnn(drug_mech,
                                  self.drug_graph_edge_index,
                                  self.drug_graph_weights)


# ============================================================
# SECTION 8: LOSS FUNCTIONS
# ============================================================

def contrastive_loss(z1, z2, temperature=0.07):
    """InfoNCE contrastive loss"""
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    logits = torch.matmul(z1, z2.T) / temperature
    labels = torch.arange(z1.size(0), device=z1.device)
    loss_fwd = F.cross_entropy(logits, labels)
    loss_rev = F.cross_entropy(logits.T, labels)
    return (loss_fwd + loss_rev) / 2


def graph_regularization_loss(embeddings, edge_index, edge_weights=None):
    """
    Graph regularization loss - encourages connected nodes to have similar embeddings
    Based on Laplacian regularization [citation:2]
    """
    # Check if graph has edges
    if edge_index is None or edge_index.shape[1] == 0:
        return torch.tensor(0.0, device=embeddings.device, requires_grad=True)

    # Ensure edge_index is on the same device as embeddings
    edge_index = edge_index.to(embeddings.device)

    # Check if indices are within bounds
    num_nodes = embeddings.shape[0]
    max_idx = edge_index.max().item()
    if max_idx >= num_nodes:
        # Filter out invalid edges
        valid_mask = (edge_index[0] < num_nodes) & (edge_index[1] < num_nodes)
        edge_index = edge_index[:, valid_mask]

        if edge_index.shape[1] == 0:
            return torch.tensor(0.0, device=embeddings.device, requires_grad=True)

    source = edge_index[0]
    target = edge_index[1]

    # Compute difference between connected nodes
    diff = embeddings[source] - embeddings[target]
    loss = (diff ** 2).sum(dim=1).mean()

    return loss

# ============================================================
# SECTION 9: DATASET CLASS
# ============================================================

class RegimenDataset(Dataset):
    def __init__(self, df, clinical_features, drug_to_idx, drug_embeddings, embed_dim=full_embed_dim):
        self.df = df.reset_index(drop=True)
        self.clinical_features = clinical_features
        self.drug_to_idx = drug_to_idx
        self.drug_embeddings = drug_embeddings
        self.embed_dim = embed_dim
        self.num_drugs = len(drug_to_idx)

        self.labels = []
        for _, row in self.df.iterrows():
            label = encode_regimen_multi_hot(row["merged_drugs_norm"], drug_to_idx)
            self.labels.append(label)

        self.mech_tensors = []
        self.mech_masks = []
        for _, row in self.df.iterrows():
            tensor, mask = build_regimen_mechanism_tensor(row["merged_drugs_norm"], drug_embeddings)
            self.mech_tensors.append(tensor)
            self.mech_masks.append(mask)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'clinical': torch.tensor(self.clinical_features[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32),
            'mech_tensor': self.mech_tensors[idx],
            'mech_mask': self.mech_masks[idx]
        }


def build_regimen_mechanism_tensor(combo, drug_embeddings, max_drugs=10, embed_dim=full_embed_dim):
    drugs = split_combo(combo)[:max_drugs]
    tensor = torch.zeros(max_drugs, embed_dim)
    for i, drug in enumerate(drugs):
        emb = drug_embeddings.get(drug, torch.zeros(embed_dim))
        tensor[i] = emb
    mask = torch.zeros(max_drugs, dtype=torch.bool)
    mask[:len(drugs)] = True
    return tensor, mask


MAX_DRUGS = 10

train_dataset = RegimenDataset(train_df, X_train, drug_to_idx, drug_embeddings)
val_dataset = RegimenDataset(val_df, X_val, drug_to_idx, drug_embeddings)
test_dataset = RegimenDataset(test_df, X_test, drug_to_idx, drug_embeddings)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


# ============================================================
# SECTION 10: MAPCT-v10 TRAINER
# ============================================================

class MAPCTv10Trainer:
    def __init__(self, model, device, drug_vocab, drug_to_idx, drug_graph_edge_index):
        self.model = model.to(device)
        self.device = device
        self.drug_vocab = drug_vocab
        self.drug_to_idx = drug_to_idx
        self.idx_to_drug = {v: k for k, v in drug_to_idx.items()}
        self.drug_graph_edge_index = drug_graph_edge_index.to(device)

        # Optimizer
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-3, weight_decay=1e-5)

        # Loss weights
        self.alpha_contrast = 0.3
        self.beta_drug = 1.0
        self.gamma_graph = 0.01

        self.train_losses = []
        self.val_losses = []

    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0
        total_contrast = 0
        total_drug = 0
        total_graph = 0

        for batch in tqdm(train_loader, desc="Training"):
            clinical = batch['clinical'].to(self.device)
            labels = batch['label'].to(self.device)
            mech_tensor = batch['mech_tensor'].to(self.device)
            mech_mask = batch['mech_mask'].to(self.device)

            logits, z_clin, z_reg = self.model(clinical, mech_tensor, mech_mask)

            # Drug prediction loss
            loss_drug = F.binary_cross_entropy_with_logits(logits, labels)

            # Contrastive loss
            loss_contrast = contrastive_loss(z_clin, z_reg)

            # Graph regularization loss (with error handling)
            try:
                drug_embeddings = self.model.get_drug_embeddings()
                loss_graph = graph_regularization_loss(drug_embeddings, self.drug_graph_edge_index)
            except Exception as e:
                print(f"⚠️ Graph loss failed: {e}. Using zero loss.")
                loss_graph = torch.tensor(0.0, device=self.device)

            # Combined loss
            loss = (self.beta_drug * loss_drug +
                  self.alpha_contrast * loss_contrast +
                  self.gamma_graph * loss_graph)

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()

            total_loss += loss.item()
            total_contrast += loss_contrast.item()
            total_drug += loss_drug.item()
            total_graph += loss_graph.item()

        n_batches = len(train_loader)
        return {
            'loss': total_loss / n_batches,
            'contrast': total_contrast / n_batches,
            'drug': total_drug / n_batches,
            'graph': total_graph / n_batches
        }

    def validate(self, val_loader):
        self.model.eval()
        total_contrast = 0
        total_drug_acc = 0

        with torch.no_grad():
            for batch in val_loader:
                clinical = batch['clinical'].to(self.device)
                labels = batch['label'].to(self.device)
                mech_tensor = batch['mech_tensor'].to(self.device)
                mech_mask = batch['mech_mask'].to(self.device)

                logits, z_clin, z_reg = self.model(clinical, mech_tensor, mech_mask)

                loss_contrast = contrastive_loss(z_clin, z_reg)
                total_contrast += loss_contrast.item()

                preds = (torch.sigmoid(logits) > 0.5).float()
                total_drug_acc += (preds == labels).float().mean().item()

        n_batches = len(val_loader)
        return {
            'contrast_loss': total_contrast / n_batches,
            'drug_accuracy': total_drug_acc / n_batches
        }

    def train(self, train_loader, val_loader, epochs=50, patience=10):
        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(epochs):
            train_metrics = self.train_epoch(train_loader)
            val_metrics = self.validate(val_loader)

            self.train_losses.append(train_metrics['loss'])

            if (epoch + 1) % 5 == 0:
                print(f"\nEpoch {epoch+1}/{epochs}")
                print(f"  Train - Loss: {train_metrics['loss']:.4f}, Contrast: {train_metrics['contrast']:.4f}, Drug: {train_metrics['drug']:.4f}")
                print(f"  Valid - Contrast: {val_metrics['contrast_loss']:.4f}, Drug Acc: {val_metrics['drug_accuracy']:.4f}")

            if val_metrics['contrast_loss'] < best_val_loss:
                best_val_loss = val_metrics['contrast_loss']
                patience_counter = 0
                self.save_checkpoint('best_gnn_model.pt')
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

        return self.train_losses

    def save_checkpoint(self, path):
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'drug_to_idx': self.drug_to_idx,
            'idx_to_drug': self.idx_to_drug,
        }, path)
        print(f"✅ Model saved to {path}")

    def load_checkpoint(self, path):
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        print(f"✅ Model loaded from {path}")


# ============================================================
# SECTION 11: VISUALIZATION FUNCTIONS (FIXED – all missing functions added)
# ============================================================

def evaluate_drug_predictions(trainer, test_loader, threshold=0.3):
    """Evaluate drug-level predictions using multi-label metrics"""
    print("\n" + "="*70)
    print("📊 DRUG PREDICTION EVALUATION")
    print("="*70)

    all_preds = []
    all_labels = []

    trainer.model.eval()
    with torch.no_grad():
        for batch in test_loader:
            clinical = batch['clinical'].to(device)
            labels = batch['label'].cpu().numpy()
            mech_tensor = batch['mech_tensor'].to(device)
            mech_mask = batch['mech_mask'].to(device)

            logits, _, _ = trainer.model(clinical, mech_tensor, mech_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > threshold).astype(np.float32)

            all_preds.extend(preds)
            all_labels.extend(labels)

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    results = {
        'precision': precision_score(all_labels, all_preds, average='micro', zero_division=0),
        'recall': recall_score(all_labels, all_preds, average='micro', zero_division=0),
        'f1': f1_score(all_labels, all_preds, average='micro', zero_division=0),
        'accuracy': accuracy_score(all_labels.flatten(), all_preds.flatten())
    }

    print(f"\n📈 Micro-averaged Metrics:")
    print(f"   Precision: {results['precision']:.4f}")
    print(f"   Recall:    {results['recall']:.4f}")
    print(f"   F1-Score:  {results['f1']:.4f}")
    print(f"   Accuracy:  {results['accuracy']:.4f}")

    return results


def visualize_gnn_latent_spaces(trainer, test_loader, test_df):
    """Visualize clinical and regimen latent spaces using PCA"""
    print("\n" + "="*70)
    print("🎨 GNN LATENT SPACE VISUALIZATION")
    print("="*70)

    trainer.model.eval()
    z_clin_list = []
    z_reg_list = []
    hba1c_labels = []

    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            clinical = batch['clinical'].to(device)
            mech_tensor = batch['mech_tensor'].to(device)
            mech_mask = batch['mech_mask'].to(device)

            _, z_clin, z_reg = trainer.model(clinical, mech_tensor, mech_mask)

            z_clin_list.append(z_clin.cpu().numpy())
            z_reg_list.append(z_reg.cpu().numpy())

            # Get HbA1c categories for coloring
            start_idx = i * BATCH_SIZE
            end_idx = min(start_idx + BATCH_SIZE, len(test_df))
            hba1c_labels.extend(test_df['hba1c_category'].iloc[start_idx:end_idx].values)

    Z_clin = np.vstack(z_clin_list)
    Z_reg = np.vstack(z_reg_list)

    # PCA
    pca = PCA(n_components=2, random_state=SEED)
    Z_clin_pca = pca.fit_transform(Z_clin)
    Z_reg_pca = pca.fit_transform(Z_reg)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    unique_cats = np.unique(hba1c_labels)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_cats)))
    for cat, color in zip(unique_cats, colors):
        idx = [j for j, c in enumerate(hba1c_labels) if c == cat]
        axes[0].scatter(Z_clin_pca[idx, 0], Z_clin_pca[idx, 1], label=cat, alpha=0.6, s=20, color=color)
    axes[0].set_title('Clinical Latent Space')
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(Z_reg_pca[:, 0], Z_reg_pca[:, 1], alpha=0.6, s=20, c='steelblue')
    axes[1].set_title('Regimen Latent Space')
    axes[1].set_xlabel('PC1')
    axes[1].set_ylabel('PC2')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('MAPCT-v10 Latent Spaces', fontsize=14)
    plt.tight_layout()
    plt.show()

    return Z_clin, Z_reg


def visualize_drug_embeddings_with_gnn(trainer, drug_vocab, drug_embeddings):
    """Visualize GNN-enhanced drug embeddings"""
    print("\n" + "="*70)
    print("🔬 GNN-ENHANCED DRUG EMBEDDINGS")
    print("="*70)

    # Get embeddings for top 30 drugs
    top_n = min(30, len(drug_vocab))
    top_drugs = drug_vocab[:top_n]
    emb_list = [drug_embeddings[d] for d in top_drugs]
    emb_matrix = torch.stack(emb_list).numpy()

    # PCA
    pca = PCA(n_components=2, random_state=SEED)
    emb_pca = pca.fit_transform(emb_matrix)

    plt.figure(figsize=(12, 10))
    plt.scatter(emb_pca[:, 0], emb_pca[:, 1], s=100, alpha=0.7)
    for i, drug in enumerate(top_drugs):
        plt.annotate(drug, (emb_pca[i, 0], emb_pca[i, 1]), fontsize=8, alpha=0.7)
    plt.title('GNN-Enhanced Drug Mechanism Embeddings')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return emb_pca


def plot_training_history(trainer):
    """Plot training loss history"""
    print("\n" + "="*70)
    print("📈 TRAINING HISTORY")
    print("="*70)

    plt.figure(figsize=(10, 6))
    plt.plot(trainer.train_losses, 'b-', linewidth=2, label='Training Loss')
    if hasattr(trainer, 'val_losses') and trainer.val_losses:
        plt.plot(trainer.val_losses, 'r-', linewidth=2, label='Validation Contrast Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Progress')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    print(f"\n📊 Final Training Loss: {trainer.train_losses[-1]:.4f}")


def visualize_drug_graph_structure(drug_edge_index, drug_vocab, drug_embeddings, drug_to_idx, top_n=30):
    """Visualize the drug similarity graph structure"""
    print("\n" + "="*70)
    print("🕸️ DRUG SIMILARITY GRAPH STRUCTURE")
    print("="*70)

    import networkx as nx
    from torch_geometric.utils import to_networkx

    # Get top N drugs (using first N from vocabulary)
    top_drugs = drug_vocab[:top_n]
    top_indices = [drug_to_idx[d] for d in top_drugs if d in drug_to_idx]

    # Create subgraph
    node_mask = torch.zeros(len(drug_vocab), dtype=torch.bool)
    node_mask[top_indices] = True

    # Filter edges for top drugs
    edge_mask = node_mask[drug_edge_index[0]] & node_mask[drug_edge_index[1]]
    sub_edge_index = drug_edge_index[:, edge_mask]

    # Convert to networkx
    G = nx.Graph()
    for i, drug in enumerate(top_drugs):
        G.add_node(drug)

    for i in range(sub_edge_index.shape[1]):
        src_idx = sub_edge_index[0, i].item()
        dst_idx = sub_edge_index[1, i].item()
        if src_idx < len(top_drugs) and dst_idx < len(top_drugs):
            src = top_drugs[src_idx]
            dst = top_drugs[dst_idx]
            if src != dst:
                G.add_edge(src, dst)

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    # Graph layout
    pos = nx.spring_layout(G, k=2, iterations=50, seed=SEED)

    nx.draw(G, pos, ax=axes[0], node_size=500, node_color='lightblue',
            edge_color='gray', with_labels=True, font_size=8, font_weight='bold')
    axes[0].set_title(f'Drug Similarity Graph (Top {top_n} Drugs)', fontsize=12)

    # Degree distribution
    degrees = [d for n, d in G.degree()]
    axes[1].hist(degrees, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
    axes[1].set_xlabel('Degree (Number of Similar Drugs)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Drug Graph Degree Distribution')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"   Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    print(f"   Average degree: {sum(degrees)/len(degrees):.2f}")

    return G


def visualize_latent_alignment(trainer, test_loader, test_df, batch_size=32):
    """Visualize how well clinical and regimen latents align using PCA and correlation"""
    print("\n" + "="*70)
    print("🔄 CLINICAL VS REGIMEN LATENT ALIGNMENT")
    print("="*70)

    trainer.model.eval()
    z_clin_list = []
    z_reg_list = []

    with torch.no_grad():
        for batch in test_loader:
            clinical = batch['clinical'].to(device)
            mech_tensor = batch['mech_tensor'].to(device)
            mech_mask = batch['mech_mask'].to(device)

            _, z_clin, z_reg = trainer.model(clinical, mech_tensor, mech_mask)

            z_clin_list.append(z_clin.cpu().numpy())
            z_reg_list.append(z_reg.cpu().numpy())

    Z_clin = np.vstack(z_clin_list)
    Z_reg = np.vstack(z_reg_list)

    # PCA
    pca = PCA(n_components=2, random_state=SEED)
    Z_clin_pca = pca.fit_transform(Z_clin)
    Z_reg_pca = pca.fit_transform(Z_reg)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].scatter(Z_clin_pca[:, 0], Z_clin_pca[:, 1], alpha=0.6, s=30, c='steelblue')
    axes[0].set_title('Clinical Latent Space', fontsize=12)
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(Z_reg_pca[:, 0], Z_reg_pca[:, 1], alpha=0.6, s=30, c='coral')
    axes[1].set_title('Regimen Latent Space', fontsize=12)
    axes[1].set_xlabel('PC1')
    axes[1].set_ylabel('PC2')
    axes[1].grid(True, alpha=0.3)

    axes[2].scatter(Z_clin_pca[:, 0], Z_reg_pca[:, 0], alpha=0.6, s=30, c='green')
    min_val = min(Z_clin_pca[:, 0].min(), Z_reg_pca[:, 0].min())
    max_val = max(Z_clin_pca[:, 0].max(), Z_reg_pca[:, 0].max())
    axes[2].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5, label='Perfect alignment')
    axes[2].set_xlabel('Clinical PC1')
    axes[2].set_ylabel('Regimen PC1')
    axes[2].set_title('Contrastive Alignment (PC1)', fontsize=12)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    # Calculate correlation
    correlation = np.corrcoef(Z_clin_pca[:, 0], Z_reg_pca[:, 0])[0, 1]
    print(f"   Correlation between Clinical and Regimen PC1: {correlation:.4f}")

    plt.suptitle('Latent Space Alignment via Contrastive Learning', fontsize=14)
    plt.tight_layout()
    plt.show()

    return correlation


def validate_graph_with_cooccurrence(train_df, drug_edge_index, drug_edge_weights, drug_vocab, drug_to_idx):
    """Compare graph similarity with actual drug co-occurrence in prescriptions"""
    print("\n" + "="*70)
    print("📊 VALIDATING GRAPH WITH DRUG CO-OCCURRENCE")
    print("="*70)

    # Calculate drug co-occurrence from training data
    cooccurrence = np.zeros((len(drug_vocab), len(drug_vocab)))

    for combo in train_df['merged_drugs_norm']:
        drugs = split_combo(combo)
        drug_indices = [drug_to_idx[d] for d in drugs if d in drug_to_idx]

        for i in range(len(drug_indices)):
            for j in range(i+1, len(drug_indices)):
                cooccurrence[drug_indices[i], drug_indices[j]] += 1
                cooccurrence[drug_indices[j], drug_indices[i]] += 1

    # Get graph similarity
    graph_similarity = torch.zeros(len(drug_vocab), len(drug_vocab))
    for i in range(drug_edge_index.shape[1]):
        src = drug_edge_index[0, i].item()
        dst = drug_edge_index[1, i].item()
        if drug_edge_weights is not None:
            graph_similarity[src, dst] = drug_edge_weights[i].item()
        else:
            graph_similarity[src, dst] = 1.0

    # Plot correlation
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Scatter plot
    graph_vals = []
    cooccur_vals = []
    for i in range(len(drug_vocab)):
        for j in range(i+1, len(drug_vocab)):
            if graph_similarity[i, j] > 0:
                graph_vals.append(graph_similarity[i, j].item() if torch.is_tensor(graph_similarity[i, j]) else graph_similarity[i, j])
                cooccur_vals.append(cooccurrence[i, j])

    axes[0].scatter(graph_vals, cooccur_vals, alpha=0.5, s=20)
    axes[0].set_xlabel('Graph Similarity (Mechanism-Based)')
    axes[0].set_ylabel('Drug Co-occurrence in Prescriptions')
    axes[0].set_title('Graph Validation: Mechanism vs Real Co-prescription')
    axes[0].grid(True, alpha=0.3)

    # Correlation
    corr = np.corrcoef(graph_vals, cooccur_vals)[0, 1]
    axes[0].text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=axes[0].transAxes,
                 fontsize=12, verticalalignment='top')

    # Bar plot of top co-occurring pairs
    top_pairs = sorted([(i, j, cooccurrence[i, j]) for i in range(len(drug_vocab))
                        for j in range(i+1, len(drug_vocab))],
                       key=lambda x: x[2], reverse=True)[:10]

    pair_names = [f"{drug_vocab[i]} + {drug_vocab[j]}" for i, j, _ in top_pairs]
    pair_cooc = [cooc for _, _, cooc in top_pairs]

    axes[1].barh(pair_names, pair_cooc, color='steelblue')
    axes[1].set_xlabel('Co-occurrence Count')
    axes[1].set_title('Top 10 Most Common Drug Pairs')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"   Correlation between mechanism similarity and co-prescription: {corr:.4f}")

    return corr


def plot_training_dashboard(trainer):
    """Create a comprehensive training dashboard with multiple metrics"""
    print("\n" + "="*70)
    print("📊 TRAINING DYNAMICS DASHBOARD")
    print("="*70)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Loss curve
    axes[0, 0].plot(trainer.train_losses, 'b-', linewidth=2, label='Training Loss')
    if hasattr(trainer, 'val_losses') and trainer.val_losses:
        axes[0, 0].plot(trainer.val_losses, 'r-', linewidth=2, label='Validation Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss Curves')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Loss improvement rate
    if len(trainer.train_losses) > 1:
        improvements = np.diff(trainer.train_losses)
        axes[0, 1].plot(improvements, 'g-', linewidth=2)
        axes[0, 1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Loss Improvement')
        axes[0, 1].set_title('Training Convergence Rate')
        axes[0, 1].grid(True, alpha=0.3)

    # Final loss info
    axes[1, 0].axis('off')
    axes[1, 0].text(0.1, 0.5, f'Final Training Loss: {trainer.train_losses[-1]:.4f}\n'
                    f'Best Validation Loss: {min(trainer.val_losses) if hasattr(trainer, "val_losses") and trainer.val_losses else "N/A"}',
                   fontsize=12, verticalalignment='center')
    axes[1, 0].set_title('Training Summary')

    # Convergence info
    axes[1, 1].axis('off')
    if len(trainer.train_losses) > 10:
        recent_improvement = (trainer.train_losses[-10] - trainer.train_losses[-1]) / trainer.train_losses[-10] * 100
        axes[1, 1].text(0.1, 0.5, f'Last 10 epochs improvement: {recent_improvement:.2f}%',
                       fontsize=12, verticalalignment='center')
    axes[1, 1].set_title('Convergence Analysis')

    plt.suptitle('MAPCT-v10 Training Dashboard', fontsize=14)
    plt.tight_layout()
    plt.show()


def visualize_latent_spaces_pca_tsne(trainer, test_loader, test_df, device, max_samples=500):
    """PCA and t‑SNE of clinical and regimen latent spaces (with UMAP optional)"""
    from sklearn.manifold import TSNE
    try:
        import umap
        UMAP_AVAILABLE = True
    except ImportError:
        UMAP_AVAILABLE = False
        print("⚠️ UMAP not installed. Install with: !pip install umap-learn")

    trainer.model.eval()
    z_clin_list = []
    z_reg_list = []
    hba1c_labels = []

    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            if len(z_clin_list) * test_loader.batch_size >= max_samples:
                break
            clinical = batch['clinical'].to(device)
            mech_tensor = batch['mech_tensor'].to(device)
            mech_mask = batch['mech_mask'].to(device)
            _, z_clin, z_reg = trainer.model(clinical, mech_tensor, mech_mask)
            z_clin_list.append(z_clin.cpu().numpy())
            z_reg_list.append(z_reg.cpu().numpy())
            start = i * test_loader.batch_size
            end = start + len(z_clin)
            hba1c_labels.extend(test_df['hba1c_category'].iloc[start:end].values)

    Z_clin = np.vstack(z_clin_list)[:max_samples]
    Z_reg = np.vstack(z_reg_list)[:max_samples]
    hba1c_labels = np.array(hba1c_labels)[:max_samples]
    unique_cats = np.unique(hba1c_labels)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_cats)))
    cat_to_color = {cat: colors[i] for i, cat in enumerate(unique_cats)}

    # ----- PCA -----
    pca = PCA(n_components=2, random_state=SEED)
    Z_clin_pca = pca.fit_transform(Z_clin)
    Z_reg_pca = pca.fit_transform(Z_reg)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for cat, color in cat_to_color.items():
        idx = np.where(hba1c_labels == cat)[0]
        axes[0].scatter(Z_clin_pca[idx, 0], Z_clin_pca[idx, 1], label=cat, alpha=0.6, s=20, color=color)
    axes[0].set_title('Clinical Latent Space (PCA)')
    axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].scatter(Z_reg_pca[:, 0], Z_reg_pca[:, 1], alpha=0.6, s=20, c='steelblue')
    axes[1].set_title('Regimen Latent Space (PCA)')
    axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    axes[1].grid(True)
    plt.tight_layout()
    plt.show()

    # ----- t‑SNE -----
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, max_iter=1000)
    Z_clin_tsne = tsne.fit_transform(Z_clin)
    Z_reg_tsne = tsne.fit_transform(Z_reg)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for cat, color in cat_to_color.items():
        idx = np.where(hba1c_labels == cat)[0]
        axes[0].scatter(Z_clin_tsne[idx, 0], Z_clin_tsne[idx, 1], label=cat, alpha=0.6, s=20, color=color)
    axes[0].set_title('Clinical Latent Space (t‑SNE)')
    axes[0].set_xlabel('t‑SNE 1')
    axes[0].set_ylabel('t‑SNE 2')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].scatter(Z_reg_tsne[:, 0], Z_reg_tsne[:, 1], alpha=0.6, s=20, c='steelblue')
    axes[1].set_title('Regimen Latent Space (t‑SNE)')
    axes[1].set_xlabel('t‑SNE 1')
    axes[1].set_ylabel('t‑SNE 2')
    axes[1].grid(True)
    plt.tight_layout()
    plt.show()

    # ----- UMAP (optional) -----
    if UMAP_AVAILABLE:
        import umap
        reducer = umap.UMAP(random_state=SEED, n_components=2)
        Z_clin_umap = reducer.fit_transform(Z_clin)
        Z_reg_umap = reducer.fit_transform(Z_reg)

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        for cat, color in cat_to_color.items():
            idx = np.where(hba1c_labels == cat)[0]
            axes[0].scatter(Z_clin_umap[idx, 0], Z_clin_umap[idx, 1], label=cat, alpha=0.6, s=20, color=color)
        axes[0].set_title('Clinical Latent Space (UMAP)')
        axes[0].set_xlabel('UMAP 1')
        axes[0].set_ylabel('UMAP 2')
        axes[0].legend()
        axes[0].grid(True)

        axes[1].scatter(Z_reg_umap[:, 0], Z_reg_umap[:, 1], alpha=0.6, s=20, c='steelblue')
        axes[1].set_title('Regimen Latent Space (UMAP)')
        axes[1].set_xlabel('UMAP 1')
        axes[1].set_ylabel('UMAP 2')
        axes[1].grid(True)
        plt.tight_layout()
        plt.show()
    else:
        print("UMAP not installed – skipping.")

    return Z_clin, Z_reg


# ============================================================
# COMPLETE EVALUATION SUITE FOR MAPCT‑v10 (GNN)
# ============================================================
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics.pairwise import cosine_distances
from itertools import combinations
from tqdm import tqdm

def evaluate_mapct_v10(trainer, test_loader, drug_vocab, idx_to_drug, device,
                       threshold=0.3, num_stochastic=5):
    """
    Evaluate MAPCT‑v10 (GNN) on test set.

    Computes:
        - Drug prediction metrics (Precision, Recall, F1, Accuracy)
        - Clinical‑Drug Alignment (cosine similarity between z_clin and z_reg)
        - Generation Diversity (Jaccard distance among perturbed predictions)

    Args:
        trainer: MAPCTv10Trainer instance (with model, etc.)
        test_loader: DataLoader for test set (yields dict)
        drug_vocab: list of drug names
        idx_to_drug: dict mapping index -> drug name
        device: 'cuda' or 'cpu'
        threshold: probability threshold for drug inclusion
        num_stochastic: number of random perturbations per patient (for diversity)
    """
    trainer.model.eval()

    all_preds = []
    all_labels = []
    align_scores = []
    all_diversity_sets = []   # list of lists of drug sets per patient

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating v10"):
            clinical = batch['clinical'].to(device)
            labels = batch['label'].cpu().numpy()
            mech_tensor = batch['mech_tensor'].to(device)
            mech_mask = batch['mech_mask'].to(device)

            # ----- 1. Drug predictions (deterministic) -----
            logits, z_clin, z_reg = trainer.model(clinical, mech_tensor, mech_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > threshold).astype(int)
            all_preds.extend(preds)
            all_labels.extend(labels)

            # ----- 2. Latent alignment (cosine similarity) -----
            # z_clin and z_reg are already normalized by encoders
            sim = (z_clin * z_reg).sum(dim=1).cpu().numpy()
            align_scores.extend(sim)

            # ----- 3. Diversity: generate stochastic variations -----
            batch_size = clinical.size(0)
            for i in range(batch_size):
                z_clin_i = z_clin[i:i+1]  # (1, latent_dim)
                drug_sets = []
                for _ in range(num_stochastic):
                    z_noisy = z_clin_i + torch.randn_like(z_clin_i) * 0.1
                    logits_noisy = trainer.model.drug_decoder(z_noisy)
                    probs_noisy = torch.sigmoid(logits_noisy).cpu().numpy()[0]
                    drugs_k = {drug_vocab[j] for j, p in enumerate(probs_noisy) if p > threshold}
                    drug_sets.append(drugs_k)
                all_diversity_sets.append(drug_sets)

    # ----- Drug prediction metrics (micro) -----
    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    precision = precision_score(y_true, y_pred, average='micro', zero_division=0)
    recall    = recall_score(y_true, y_pred, average='micro', zero_division=0)
    f1        = f1_score(y_true, y_pred, average='micro', zero_division=0)
    accuracy  = accuracy_score(y_true.flatten(), y_pred.flatten())
    alignment = np.mean(align_scores)

    # ----- Generation diversity (Jaccard distance) -----
    all_jaccard_dists = []
    for patient_sets in all_diversity_sets:
        if len(patient_sets) < 2:
            continue
        for set1, set2 in combinations(patient_sets, 2):
            if len(set1 | set2) == 0:
                dist = 0.0
            else:
                jaccard = len(set1 & set2) / len(set1 | set2)
                dist = 1 - jaccard
            all_jaccard_dists.append(dist)
    diversity = np.mean(all_jaccard_dists) if all_jaccard_dists else 0.0

    # ----- Print results -----
    print("\n" + "="*50)
    print("MAPCT‑v10 (GNN) Test Evaluation")
    print("="*50)
    print(f"Threshold: {threshold}")
    print(f"Precision (micro): {precision:.4f}")
    print(f"Recall    (micro): {recall:.4f}")
    print(f"F1‑score  (micro): {f1:.4f}")
    print(f"Accuracy (overall): {accuracy:.4f}")
    print(f"Clinical‑Drug Alignment (mean cosine): {alignment:.4f}")
    print(f"Generation Diversity (mean pairwise Jaccard distance): {diversity:.4f}")

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'alignment': alignment,
        'diversity': diversity
    }




In [ ]:
# ============================================================
# JOINT LATENT SPACE VISUALIZATIONS (PCA, t‑SNE, UMAP) FOR v10
# ============================================================

def visualize_joint_pca_v10(trainer, test_loader, test_df, device,
                            max_samples=500, save_path=None):
    """
    Joint PCA projection of clinical and regimen latents.
    Circles = clinical, Squares = regimen, coloured by HbA1c category.
    Grey lines connect each patient’s clinical → regimen point.
    """
    print("\n🎨 Joint PCA: Clinical + Regimen Latent Spaces (v10)")
    trainer.model.eval()

    z_clin_list, z_reg_list, labels_list = [], [], []
    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            if len(z_clin_list) * BATCH_SIZE >= max_samples:
                break
            clinical = batch['clinical'].to(device)
            mech_tensor = batch['mech_tensor'].to(device)
            mech_mask = batch['mech_mask'].to(device)

            _, z_clin, z_reg = trainer.model(clinical, mech_tensor, mech_mask)

            z_clin_list.append(z_clin.cpu().numpy())
            z_reg_list.append(z_reg.cpu().numpy())

            start = i * BATCH_SIZE
            end = min(start + len(z_clin), len(test_df))
            labels_list.extend(test_df['hba1c_category'].iloc[start:end].values)

    Z_clin = np.vstack(z_clin_list)[:max_samples]
    Z_reg = np.vstack(z_reg_list)[:max_samples]
    labels = np.array(labels_list)[:max_samples]

    # Joint PCA
    Z_joint = np.vstack([Z_clin, Z_reg])
    pca = PCA(n_components=2, random_state=SEED)
    Z_joint_pca = pca.fit_transform(Z_joint)
    n_clin = len(Z_clin)
    Z_clin_pca = Z_joint_pca[:n_clin]
    Z_reg_pca = Z_joint_pca[n_clin:]

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    unique_labels = np.unique(labels)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
    color_map = dict(zip(unique_labels, colors))

    for label in unique_labels:
        mask = (labels == label)
        ax.scatter(Z_clin_pca[mask, 0], Z_clin_pca[mask, 1],
                   marker='o', s=40, alpha=0.6, color=color_map[label],
                   label=f'Clinical – {label}')
        ax.scatter(Z_reg_pca[mask, 0], Z_reg_pca[mask, 1],
                   marker='s', s=40, alpha=0.6, color=color_map[label],
                   label=f'Regimen – {label}')

    # Connecting lines (first 100 patients)
    for i in range(min(100, n_clin)):
        ax.plot([Z_clin_pca[i,0], Z_reg_pca[i,0]],
                [Z_clin_pca[i,1], Z_reg_pca[i,1]],
                color='gray', alpha=0.2, linewidth=0.8)

    ax.set_title('Clinical and Regimen Latent Spaces with Alignment (PCA)', fontsize=12)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.legend(ncol=2, fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    print("✅ Joint PCA plot saved.")


def visualize_joint_tsne_v10(trainer, test_loader, test_df, device,
                             max_samples=500, perplexity=30, save_path=None):
    """
    Joint t‑SNE projection of clinical and regimen latents.
    """
    print("\n🎨 Joint t‑SNE: Clinical + Regimen Latent Spaces (v10)")
    trainer.model.eval()

    z_clin_list, z_reg_list, labels_list = [], [], []
    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            if len(z_clin_list) * BATCH_SIZE >= max_samples:
                break
            clinical = batch['clinical'].to(device)
            mech_tensor = batch['mech_tensor'].to(device)
            mech_mask = batch['mech_mask'].to(device)

            _, z_clin, z_reg = trainer.model(clinical, mech_tensor, mech_mask)

            z_clin_list.append(z_clin.cpu().numpy())
            z_reg_list.append(z_reg.cpu().numpy())

            start = i * BATCH_SIZE
            end = min(start + len(z_clin), len(test_df))
            labels_list.extend(test_df['hba1c_category'].iloc[start:end].values)

    Z_clin = np.vstack(z_clin_list)[:max_samples]
    Z_reg = np.vstack(z_reg_list)[:max_samples]
    labels = np.array(labels_list)[:max_samples]

    # Joint t‑SNE
    Z_joint = np.vstack([Z_clin, Z_reg])
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=perplexity, max_iter=1000)
    Z_joint_tsne = tsne.fit_transform(Z_joint)
    n_clin = len(Z_clin)
    Z_clin_tsne = Z_joint_tsne[:n_clin]
    Z_reg_tsne = Z_joint_tsne[n_clin:]

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    unique_labels = np.unique(labels)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
    color_map = dict(zip(unique_labels, colors))

    for label in unique_labels:
        mask = (labels == label)
        ax.scatter(Z_clin_tsne[mask, 0], Z_clin_tsne[mask, 1],
                   marker='o', s=40, alpha=0.6, color=color_map[label],
                   label=f'Clinical – {label}')
        ax.scatter(Z_reg_tsne[mask, 0], Z_reg_tsne[mask, 1],
                   marker='s', s=40, alpha=0.6, color=color_map[label],
                   label=f'Regimen – {label}')

    # Connecting lines
    for i in range(min(100, n_clin)):
        ax.plot([Z_clin_tsne[i,0], Z_reg_tsne[i,0]],
                [Z_clin_tsne[i,1], Z_reg_tsne[i,1]],
                color='gray', alpha=0.2, linewidth=0.8)

    ax.set_title('Clinical and Regimen Latent Spaces with Alignment (t‑SNE)', fontsize=12)
    ax.set_xlabel('t‑SNE 1')
    ax.set_ylabel('t‑SNE 2')
    ax.legend(ncol=2, fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    print("✅ Joint t‑SNE plot saved.")


def visualize_joint_umap_v10(trainer, test_loader, test_df, device,
                             max_samples=500, n_neighbors=15, min_dist=0.1,
                             save_path=None):
    """
    Joint UMAP projection of clinical and regimen latents.
    Requires umap-learn installed.
    """
    if not UMAP_AVAILABLE:
        print("⚠️ UMAP not installed – skipping joint UMAP plot.")
        return

    print("\n🎨 Joint UMAP: Clinical + Regimen Latent Spaces (v10)")
    trainer.model.eval()

    z_clin_list, z_reg_list, labels_list = [], [], []
    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            if len(z_clin_list) * BATCH_SIZE >= max_samples:
                break
            clinical = batch['clinical'].to(device)
            mech_tensor = batch['mech_tensor'].to(device)
            mech_mask = batch['mech_mask'].to(device)

            _, z_clin, z_reg = trainer.model(clinical, mech_tensor, mech_mask)

            z_clin_list.append(z_clin.cpu().numpy())
            z_reg_list.append(z_reg.cpu().numpy())

            start = i * BATCH_SIZE
            end = min(start + len(z_clin), len(test_df))
            labels_list.extend(test_df['hba1c_category'].iloc[start:end].values)

    Z_clin = np.vstack(z_clin_list)[:max_samples]
    Z_reg = np.vstack(z_reg_list)[:max_samples]
    labels = np.array(labels_list)[:max_samples]

    # Joint UMAP
    Z_joint = np.vstack([Z_clin, Z_reg])
    reducer = umap.UMAP(random_state=SEED, n_components=2,
                        n_neighbors=n_neighbors, min_dist=min_dist)
    Z_joint_umap = reducer.fit_transform(Z_joint)
    n_clin = len(Z_clin)
    Z_clin_umap = Z_joint_umap[:n_clin]
    Z_reg_umap = Z_joint_umap[n_clin:]

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    unique_labels = np.unique(labels)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
    color_map = dict(zip(unique_labels, colors))

    for label in unique_labels:
        mask = (labels == label)
        ax.scatter(Z_clin_umap[mask, 0], Z_clin_umap[mask, 1],
                   marker='o', s=40, alpha=0.6, color=color_map[label],
                   label=f'Clinical – {label}')
        ax.scatter(Z_reg_umap[mask, 0], Z_reg_umap[mask, 1],
                   marker='s', s=40, alpha=0.6, color=color_map[label],
                   label=f'Regimen – {label}')

    # Connecting lines
    for i in range(min(100, n_clin)):
        ax.plot([Z_clin_umap[i,0], Z_reg_umap[i,0]],
                [Z_clin_umap[i,1], Z_reg_umap[i,1]],
                color='gray', alpha=0.2, linewidth=0.8)

    ax.set_title('Clinical and Regimen Latent Spaces with Alignment (UMAP)', fontsize=12)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.legend(ncol=2, fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    print("✅ Joint UMAP plot saved.")

In [ ]:
# ============================================================
# SECTION 12: MAIN EXECUTION (CORRECTED)
# ============================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🚀 MAPCT-v10: GRAPH NEURAL NETWORK FOR REGIMEN GENERATION")
    print("="*80)

    # Initialize model
    clin_dim = X_train.shape[1]
    model = MAPCTv10GNN(
        clin_dim=clin_dim,
        num_drugs=num_drugs,
        drug_graph_edge_index=drug_edge_index,
        drug_graph_weights=drug_edge_weights,
        latent_dim=128,
        gnn_type='gat',
        n_gnn_layers=3
    ).to(device)

    print(f"\n✅ Model initialized:")
    print(f"   Clinical input dim: {clin_dim}")
    print(f"   Drug vocabulary size: {num_drugs}")
    print(f"   GNN type: GAT (Graph Attention Network)")
    print(f"   GNN layers: 3")
    print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Create trainer
    trainer = MAPCTv10Trainer(model, device, drug_vocab, drug_to_idx, drug_edge_index)

    # Train model
    print("\n" + "="*80)
    print("🎯 STARTING GNN TRAINING")
    print("="*80)

    history = trainer.train(train_loader, val_loader, epochs=50, patience=10)

    # Load best model
    trainer.load_checkpoint('best_gnn_model.pt')

    # ============================================================
    # COMPLETE EVALUATION (alignment & diversity + drug metrics)
    # ============================================================
    results_full = evaluate_mapct_v10(
        trainer=trainer,
        test_loader=test_loader,
        drug_vocab=drug_vocab,
        idx_to_drug=idx_to_drug,
        device=device,
        threshold=0.3,
        num_stochastic=5
    )
    print("\nFull evaluation results (alignment & diversity):", results_full)

    # ============================================================
    # EXISTING VISUALIZATIONS
    # ============================================================
    # Visualize latent spaces (PCA)
    Z_clin, Z_reg = visualize_gnn_latent_spaces(trainer, test_loader, test_df)

    # Visualize GNN-enhanced drug embeddings
    gnn_embeddings = visualize_drug_embeddings_with_gnn(trainer, drug_vocab, drug_embeddings)

    # Plot training history
    plot_training_history(trainer)

    # ============================================================
    # PCA + t‑SNE + UMAP OF CLINICAL AND REGIMEN LATENT SPACES
    # ============================================================
    print("\n" + "="*80)
    print("🎨 PCA, t‑SNE & UMAP OF CLINICAL AND REGIMEN LATENT SPACES")
    print("="*80)
    Z_clin, Z_reg = visualize_latent_spaces_pca_tsne(trainer, test_loader, test_df, device, max_samples=500)

    # ============================================================
    # ADDITIONAL VISUALIZATIONS
    # ============================================================
    print("\n" + "="*80)
    print("📊 GENERATING ADDITIONAL VISUALIZATIONS")
    print("="*80)

    # 1. Drug graph structure visualization
    try:
        G = visualize_drug_graph_structure(drug_edge_index, drug_vocab, drug_embeddings, drug_to_idx, top_n=30)
    except Exception as e:
        print(f"⚠️ Could not generate drug graph: {e}")

    # 2. Latent space alignment visualization (PC1 correlation)
    try:
        alignment_corr = visualize_latent_alignment(trainer, test_loader, test_df, batch_size=BATCH_SIZE)
    except Exception as e:
        print(f"⚠️ Could not generate latent alignment: {e}")

    # 3. Validate graph with co-occurrence
    try:
        graph_corr = validate_graph_with_cooccurrence(train_df, drug_edge_index, drug_edge_weights, drug_vocab, drug_to_idx)
    except Exception as e:
        print(f"⚠️ Could not validate graph: {e}")

    # 4. Training dashboard
    try:
        plot_training_dashboard(trainer)
    except Exception as e:
        print(f"⚠️ Could not generate training dashboard: {e}")

    # ========== JOINT LATENT SPACE VISUALIZATIONS ==========
    print("\n🎨 Generating joint latent space plots (PCA, t‑SNE, UMAP)...")
    visualize_joint_pca_v10(trainer, test_loader, test_df, device,
                            max_samples=500, save_path='joint_pca_v10.png')
    visualize_joint_tsne_v10(trainer, test_loader, test_df, device,
                            max_samples=500, perplexity=30, save_path='joint_tsne_v10.png')
    if UMAP_AVAILABLE:
        visualize_joint_umap_v10(trainer, test_loader, test_df, device,
                                max_samples=500, n_neighbors=15, min_dist=0.1,
                                save_path='joint_umap_v10.png')

    # Save final model
    trainer.save_checkpoint('/content/drive/MyDrive/khezri/mapct_v10_model.pt')

    print("\n" + "="*80)
    print("✅ MAPCT-v10 COMPLETE!")
    print("="*80)